# 차량 파손 YOLO 세그멘테이션 학습 (Colab)

Google Drive에 올린 `yolo_dataset_sample_95k_colab.zip`을 Colab 런타임으로 복사하고, 로컬 런타임 디스크에서 데이터셋을 읽어 YOLOv8n-seg를 학습합니다. 학습 결과와 epoch별 체크포인트는 Drive에 저장합니다.

## 실행 순서
1. Colab 메뉴에서 **런타임 → 런타임 유형 변경 → T4 GPU**(또는 사용 가능한 GPU)를 선택합니다.
2. ZIP을 Google Drive의 `내 드라이브` 최상위에 올립니다. 다른 폴더에 올렸다면 설정 셀의 `ZIP_IN_DRIVE` 경로를 바꿉니다.
3. 위에서부터 셀을 차례대로 실행하고 Drive 접근 권한을 허용합니다.
4. 학습 중 런타임이 끊겼다면 설정 셀에서 `RESUME = True`로 바꾸고 같은 실행 폴더의 `last.pt`부터 이어 학습합니다.

학습 시간은 Colab GPU 종류와 사용 가능 시간에 따라 달라집니다. 118,000장 전체를 10 epoch 학습하므로 오래 걸릴 수 있습니다.

In [1]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU를 사용할 수 없습니다. Colab에서 런타임 유형을 GPU로 바꾸고 "
        "이 셀부터 다시 실행하세요."
    )

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}")

GPU: Tesla T4
PyTorch: 2.11.0+cu130 | CUDA: 13.0


In [2]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive")
ZIP_IN_DRIVE = DRIVE_ROOT / "yolo_dataset_sample_95k_colab.zip"
LOCAL_ZIP = Path("/content/yolo_dataset_sample_95k_colab.zip")
DATASET_DIR = Path("/content/yolo_dataset_sample_95k")
DATA_YAML = DATASET_DIR / "data.yaml"

if not ZIP_IN_DRIVE.is_file():
    matches = list(DRIVE_ROOT.glob("*95k*.zip"))
    raise FileNotFoundError(
        f"Drive에서 ZIP을 찾을 수 없습니다: {ZIP_IN_DRIVE}\n"
        f"내 드라이브의 95k ZIP 후보: {[p.name for p in matches]}\n"
        "ZIP이 하위 폴더에 있으면 ZIP_IN_DRIVE를 해당 경로로 수정하세요."
    )

print(f"Drive ZIP: {ZIP_IN_DRIVE}")

Mounted at /content/drive
Drive ZIP: /content/drive/MyDrive/yolo_dataset_sample_95k_colab.zip


In [3]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "ultralytics"],
    check=True,
)
print("Ultralytics 설치 확인 완료")

Ultralytics 설치 확인 완료


In [ ]:
import shutil
import subprocess
import zipfile
import yaml

EXPECTED_COUNTS = {
    "images/train": 95_000,
    "labels/train": 95_000,
    "images/validation": 23_000,
    "labels/validation": 23_000,
}
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

archive_size = ZIP_IN_DRIVE.stat().st_size
if archive_size == 0:
    raise ValueError(f"ZIP 파일 크기가 0입니다: {ZIP_IN_DRIVE}")

free_bytes = shutil.disk_usage("/content").free
if not LOCAL_ZIP.is_file() or LOCAL_ZIP.stat().st_size != archive_size:
    if free_bytes < archive_size + 1 * 1024**3:
        raise OSError(
            "런타임 디스크 공간이 부족합니다. ZIP 복사에 필요한 여유 공간을 확보하세요. "
            f"현재 여유: {free_bytes / 1024**3:.1f} GiB"
        )
    print(f"Drive에서 런타임 디스크로 복사 중 ({archive_size / 1e9:.2f} GB)...")
    shutil.copy2(ZIP_IN_DRIVE, LOCAL_ZIP)
else:
    print("기존 /content ZIP을 재사용합니다.")

with zipfile.ZipFile(LOCAL_ZIP) as archive:
    members = archive.infolist()
    archive_counts = {key: 0 for key in EXPECTED_COUNTS}
    for member in members:
        if member.is_dir():
            continue
        name = member.filename
        if name.startswith("images/train/") and Path(name).suffix.lower() in IMAGE_EXTENSIONS:
            archive_counts["images/train"] += 1
        elif name.startswith("labels/train/") and name.endswith(".txt"):
            archive_counts["labels/train"] += 1
        elif name.startswith("images/validation/") and Path(name).suffix.lower() in IMAGE_EXTENSIONS:
            archive_counts["images/validation"] += 1
        elif name.startswith("labels/validation/") and name.endswith(".txt"):
            archive_counts["labels/validation"] += 1
    if archive_counts != EXPECTED_COUNTS:
        raise ValueError(
            f"ZIP 내부 파일 개수가 예상과 다릅니다.\n"
            f"예상: {EXPECTED_COUNTS}\n실제: {archive_counts}"
        )
    if "data.yaml" not in archive.namelist():
        raise FileNotFoundError("ZIP 루트에 data.yaml이 없습니다.")
    uncompressed_bytes = sum(member.file_size for member in members)
    config = yaml.safe_load(archive.read("data.yaml"))

expected_names = {0: "Scratch", 1: "Dent", 2: "Breakage", 3: "Separated"}
actual_names = {int(key): value for key, value in config.get("names", {}).items()}
if actual_names != expected_names:
    raise ValueError(f"클래스 설정이 예상과 다릅니다: {actual_names}")

def split_is_ready(split):
    for kind in ("images", "labels"):
        folder = DATASET_DIR / kind / split
        expected = EXPECTED_COUNTS[f"{kind}/{split}"]
        if not folder.is_dir() or sum(path.is_file() for path in folder.iterdir()) != expected:
            return False
    return True

dataset_ready = DATA_YAML.is_file() and all(
    split_is_ready(split) for split in ("train", "validation")
)
if not dataset_ready:
    free_bytes = shutil.disk_usage("/content").free
    if free_bytes < uncompressed_bytes + 1 * 1024**3:
        raise OSError(
            "압축 해제 공간이 부족합니다. "
            f"필요 약 {uncompressed_bytes / 1024**3:.1f} GiB, "
            f"현재 여유 {free_bytes / 1024**3:.1f} GiB"
        )
    if shutil.which("unzip") is None:
        raise FileNotFoundError("런타임에서 unzip 명령을 찾을 수 없습니다.")
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    print(f"압축 해제 중: {LOCAL_ZIP} → {DATASET_DIR}")
    subprocess.run(
        ["unzip", "-q", "-o", str(LOCAL_ZIP), "-d", str(DATASET_DIR)],
        check=True,
    )
else:
    print("이미 압축 해제된 데이터셋을 재사용합니다.")

if not DATA_YAML.is_file():
    raise FileNotFoundError(f"압축 해제 결과에 data.yaml이 없습니다: {DATA_YAML}")
config["path"] = str(DATASET_DIR)
DATA_YAML.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
for split, count in (("train", 95_000), ("validation", 23_000)):
    if not split_is_ready(split):
        raise RuntimeError(f"압축 해제된 {split} 데이터 파일 수가 예상과 다릅니다.")

print(f"데이터셋 준비 완료: {DATASET_DIR}")
print(f"Train: 95,000 | Validation: 23,000 | Classes: {list(expected_names.values())}")
print(DATA_YAML.read_text(encoding="utf-8"))

Drive에서 런타임 디스크로 복사 중 (9.38 GB)...


In [ ]:
# 필요하면 이 설정만 수정하세요.
EPOCHS = 10
IMGSZ = 320
BATCH = -1  # GPU 메모리에 맞춰 자동 선택
RUN_NAME = "aihub_sample_95k"
RESUME = False  # 중단된 같은 실행을 이어갈 때 True
SEED = 42

DRIVE_RUNS = DRIVE_ROOT / "car_damage_training"
RUN_DIR = DRIVE_RUNS / RUN_NAME
LAST_CHECKPOINT = RUN_DIR / "weights" / "last.pt"
BEST_CHECKPOINT = RUN_DIR / "weights" / "best.pt"

if RESUME:
    if not LAST_CHECKPOINT.is_file():
        raise FileNotFoundError(f"이어 학습할 last.pt가 없습니다: {LAST_CHECKPOINT}")
else:
    if RUN_DIR.exists():
        raise FileExistsError(
            f"실행 폴더가 이미 있습니다: {RUN_DIR}\n"
            "이어 학습하려면 RESUME=True로 바꾸거나, 새 RUN_NAME을 지정하세요."
        )

print(f"Run directory: {RUN_DIR}")
print(f"Resume: {RESUME} | epochs={EPOCHS} | imgsz={IMGSZ} | batch={BATCH}")

In [ ]:
from ultralytics import YOLO

DRIVE_RUNS.mkdir(parents=True, exist_ok=True)
if RESUME:
    model = YOLO(str(LAST_CHECKPOINT))
    results = model.train(resume=True, device=0)
else:
    model = YOLO("yolov8n-seg.pt")
    results = model.train(
        data=str(DATA_YAML),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=0,
        workers=4,
        project=str(DRIVE_RUNS),
        name=RUN_NAME,
        exist_ok=False,
        seed=SEED,
        amp=True,
        cache=False,
        save=True,
        save_period=1,
        patience=EPOCHS,
    )

print("학습 완료")

In [ ]:
import shutil

RESULTS_CSV = RUN_DIR / "results.csv"
for label, path in (
    ("best.pt", BEST_CHECKPOINT),
    ("last.pt", LAST_CHECKPOINT),
    ("results.csv", RESULTS_CSV),
):
    print(f"{label}: {'완료' if path.is_file() else '아직 없음'} — {path}")

if BEST_CHECKPOINT.is_file():
    exported_model = DRIVE_ROOT / "car_damage_training" / "yolo_seg_95k_best.pt"
    if BEST_CHECKPOINT.resolve() != exported_model.resolve():
        shutil.copy2(BEST_CHECKPOINT, exported_model)
    print(f"최종 best 모델 복사 완료: {exported_model}")